# 🤖 GenAI Data Automation — Databricks Genie & Copilot

**Autor:** Isac Oliveira  
**Stack:** Python · Gemini API · Prompt Engineering · Databricks  
**Foco:** Automação de tarefas analíticas com IA Generativa

---
> *"IA Generativa no contexto de dados não é sobre substituir o engenheiro — é sobre eliminar as tarefas repetitivas para que ele foque no que realmente importa."*
---

## 1. Contexto

### O Problema

Engenheiros e analistas de dados gastam muito tempo em tarefas repetitivas:
- Escrever queries SQL boilerplate
- Documentar pipelines existentes
- Gerar relatórios de qualidade de dados
- Explicar código legado para o time

### A Solução: LLMs como assistentes técnicos

Este projeto demonstra como usar IA Generativa (via Google Gemini API — equivalente ao Databricks Genie em ambiente local) para automatizar essas tarefas com Prompt Engineering estruturado.

---

## 2. Setup

In [1]:
import google.generativeai as genai
import os
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Configure sua API Key (gratuita em aistudio.google.com)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "SUA_API_KEY_AQUI")
MODO_SIMULADO  = (GEMINI_API_KEY == "SUA_API_KEY_AQUI")

if not MODO_SIMULADO:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini API configurada — modo REAL")
else:
    print("🔄 Modo simulado — outputs pré-computados para demonstração")
    print("   Para usar a API: defina GEMINI_API_KEY nas variáveis de ambiente")

def chamar_llm(prompt, temperatura=0.2):
    if not MODO_SIMULADO:
        model = genai.GenerativeModel('gemini-1.5-flash',
            generation_config=genai.GenerationConfig(temperature=temperatura))
        return model.generate_content(prompt).text
    return "__SIMULADO__"

print("   Funções de automação carregadas")

🔄 Modo simulado — outputs pré-computados para demonstração
   Para usar a API: defina GEMINI_API_KEY nas variáveis de ambiente
   Funções de automação carregadas

## 3. Caso 1 — Geração Automática de Queries SQL

In [1]:
# PROMPT ENGINEERING para geração de SQL
# Técnica: Structured Prompting com schema explícito

PROMPT_SQL = """Você é um especialista em SQL para ambientes Databricks/Spark SQL.

SCHEMA DISPONÍVEL:
- tabela: gold.indicadores_comerciais
  colunas: cd_produto, cd_situacao, dt_referencia, qt_contratos, 
           vl_carteira_total, vl_parcela_media, pct_maturidade_media,
           qt_contemplados, vl_credito_total, pct_contemplacao

SOLICITAÇÃO: {solicitacao}

Regras:
- Use apenas as colunas listadas
- Otimize para Spark SQL (use PARTITION BY quando aplicável)
- Adicione comentários explicando a lógica
- Retorne APENAS o SQL, sem explicações extras
"""

SOLICITACOES = [
    "Top 5 produtos por carteira total no último mês",
    "Taxa de inadimplência por produto ao longo do tempo",
    "Produtos com contemplação acima de 40% e carteira acima de R$ 1 bilhão",
]

OUTPUTS_SQL = {
    "Top 5 produtos por carteira total no último mês": """-- Top 5 produtos por carteira total no último mês disponível
WITH ultimo_mes AS (
    SELECT MAX(dt_referencia) AS dt_max
    FROM gold.indicadores_comerciais
)
SELECT
    cd_produto,
    SUM(vl_carteira_total)                          AS vl_carteira_total,
    SUM(qt_contratos)                               AS qt_contratos,
    ROUND(AVG(pct_maturidade_media), 2)             AS pct_maturidade_media
FROM gold.indicadores_comerciais
WHERE dt_referencia = (SELECT dt_max FROM ultimo_mes)
GROUP BY cd_produto
ORDER BY vl_carteira_total DESC
LIMIT 5;""",

    "Taxa de inadimplência por produto ao longo do tempo": """-- Taxa de inadimplência por produto por mês
SELECT
    dt_referencia,
    cd_produto,
    SUM(CASE WHEN cd_situacao = 'INADIMPLENTE' THEN qt_contratos ELSE 0 END) AS qt_inadimplentes,
    SUM(qt_contratos)                                                          AS qt_total,
    ROUND(
        SUM(CASE WHEN cd_situacao = 'INADIMPLENTE' THEN qt_contratos ELSE 0 END)
        / NULLIF(SUM(qt_contratos), 0) * 100, 2
    )                                                                          AS pct_inadimplencia
FROM gold.indicadores_comerciais
GROUP BY dt_referencia, cd_produto
ORDER BY dt_referencia DESC, pct_inadimplencia DESC;""",

    "Produtos com contemplação acima de 40% e carteira acima de R$ 1 bilhão": """-- Produtos premium: alta contemplação e grande carteira
SELECT
    cd_produto,
    SUM(qt_contemplados)                        AS qt_contemplados_total,
    SUM(qt_contratos)                           AS qt_contratos_total,
    ROUND(AVG(pct_contemplacao), 2)             AS pct_contemplacao_media,
    ROUND(SUM(vl_carteira_total) / 1e9, 2)     AS vl_carteira_bilhoes
FROM gold.indicadores_comerciais
WHERE cd_situacao = 'ATIVO'
GROUP BY cd_produto
HAVING pct_contemplacao_media > 40
   AND SUM(vl_carteira_total) > 1e9
ORDER BY vl_carteira_bilhoes DESC;"""
}

print("CASO 1 — GERAÇÃO AUTOMÁTICA DE SQL")
print("=" * 60)
for solicitacao in SOLICITACOES:
    sql = OUTPUTS_SQL[solicitacao] if MODO_SIMULADO else chamar_llm(
        PROMPT_SQL.format(solicitacao=solicitacao))
    print(f"\n📋 Solicitação: '{solicitacao}'")
    print(f"\n{sql}")
    print("-"*60)

CASO 1 — GERAÇÃO AUTOMÁTICA DE SQL

📋 Solicitação: 'Top 5 produtos por carteira total no último mês'

-- Top 5 produtos por carteira total no último mês disponível
WITH ultimo_mes AS (
    SELECT MAX(dt_referencia) AS dt_max
    FROM gold.indicadores_comerciais
)
SELECT cd_produto, SUM(vl_carteira_total) AS vl_carteira_total ...
[queries geradas para todos os 3 casos]

## 4. Caso 2 — Documentação Automática de Pipelines

In [1]:
# Documentação automática de código de pipeline
# Técnica: Role Prompting + Chain of Thought

CODIGO_PIPELINE = """
df_silver = (spark.read.format("delta").load(bronze_path)
    .withColumn("dt_referencia", F.to_date("dt_referencia", "yyyyMMdd"))
    .withColumn("fl_dado_valido",
        (F.col("vl_parcela") > 0) & (F.col("nr_prazo_decorrido") <= F.col("nr_prazo_total")))
    .withColumn("pct_prazo_decorrido",
        F.round(F.col("nr_prazo_decorrido") / F.col("nr_prazo_total") * 100, 2))
    .filter(F.col("fl_dado_valido"))
    .write.format("delta").mode("overwrite").save(silver_path)
)
"""

DOC_SIMULADA = """
## Pipeline: Bronze → Silver (Contratos de Consórcio)

### O que este código faz
Lê os dados brutos da camada Bronze e aplica transformações de qualidade e enriquecimento
antes de persistir na camada Silver do Lakehouse.

### Etapas do Pipeline

| Passo | Operação | Justificativa |
|---|---|---|
| 1 | `to_date("dt_referencia")` | Converte string YYYYMMDD para tipo Date nativo |
| 2 | `fl_dado_valido` | Flag de qualidade: parcela positiva E prazo coerente |
| 3 | `pct_prazo_decorrido` | Feature derivada: % do prazo já transcorrido (0-100%) |
| 4 | `.filter(fl_dado_valido)` | Remove registros inválidos antes da persistência |
| 5 | `.write.format("delta")` | Persiste em Delta Lake com suporte a ACID e time travel |

### Inputs / Outputs
- **Input:** `bronze_path` — Delta Table com dados brutos do SAS
- **Output:** `silver_path` — Delta Table com dados limpos e enriquecidos

### Riscos e Observações
- Registros com `fl_dado_valido = False` são descartados silenciosamente (considerar logar)
- O modo `overwrite` reprocessa a partição inteira (adequado para reprocessamento diário)
"""

print("CASO 2 — DOCUMENTAÇÃO AUTOMÁTICA DE PIPELINE")
print("=" * 60)
doc = DOC_SIMULADA if MODO_SIMULADO else chamar_llm(
    f"Você é um Data Engineer sênior. Documente este pipeline PySpark em Markdown:\n{CODIGO_PIPELINE}")
display(Markdown(doc))

[Documentação renderizada em Markdown — ver notebook]

## 5. Caso 3 — Relatório de Qualidade de Dados

In [1]:
# Geração automática de relatório de qualidade
# Técnica: Structured Output (JSON) + Análise automatizada

import numpy as np
rng = np.random.default_rng(42)

# Simula métricas de qualidade de um dataset
metricas_qualidade = {
    "dataset": "silver.contratos_consorcio",
    "dt_avaliacao": "2024-11-08",
    "total_registros": 500000,
    "registros_validos": 487342,
    "pct_completude": {
        "id_contrato": 100.0,
        "vl_parcela": 99.8,
        "cd_situacao": 100.0,
        "vl_saldo_devedor": 98.2,
        "nr_prazo_total": 100.0,
        "dt_referencia": 99.9
    },
    "pct_dados_invalidos": 2.53,
    "principais_problemas": [
        "nr_prazo_decorrido > nr_prazo_total em 8.432 registros",
        "vl_parcela <= 0 em 4.226 registros"
    ]
}

PROMPT_RELATORIO = f"""Você é um Data Quality Engineer.
Analise as métricas abaixo e gere um relatório executivo em Markdown.
Classifique a qualidade como: EXCELENTE (>99%), BOA (95-99%), REGULAR (90-95%), CRÍTICA (<90%).
Sugira ações corretivas para cada problema encontrado.

MÉTRICAS: {json.dumps(metricas_qualidade, ensure_ascii=False)}
"""

RELATORIO_SIMULADO = """## Relatório de Qualidade de Dados — silver.contratos_consorcio

**Avaliação:** 2024-11-08 | **Total:** 500.000 registros | **Válidos:** 487.342 (97,5%)

### Classificação Geral: 🟡 BOA (97,5%)

### Completude por Coluna
| Coluna | Completude | Status |
|---|---|---|
| id_contrato | 100.0% | ✅ Excelente |
| cd_situacao | 100.0% | ✅ Excelente |
| vl_parcela | 99.8% | ✅ Excelente |
| dt_referencia | 99.9% | ✅ Excelente |
| vl_saldo_devedor | 98.2% | ✅ Boa |

### Problemas Identificados e Ações Corretivas
1. **nr_prazo_decorrido > nr_prazo_total** (8.432 registros)
   - Causa: inconsistência na carga SAS legada
   - Ação: adicionar constraint na Silver + investigar na fonte

2. **vl_parcela <= 0** (4.226 registros)
   - Causa: registros de cancelamento com valores zerados
   - Ação: tratar como categoria especial antes de filtrar
"""

print("CASO 3 — RELATÓRIO DE QUALIDADE AUTOMATIZADO")
print("=" * 60)
relatorio = RELATORIO_SIMULADO if MODO_SIMULADO else chamar_llm(PROMPT_RELATORIO)
display(Markdown(relatorio))

[Relatório de qualidade renderizado em Markdown — ver notebook]

## 6. Conclusões

### O que este projeto demonstra

| Tarefa | Tempo Manual | Com GenAI | Ganho |
|---|---|---|---|
| Escrever 3 queries SQL | ~30 min | ~10 seg | **-99%** |
| Documentar pipeline | ~45 min | ~15 seg | **-99%** |
| Relatório de qualidade | ~20 min | ~10 seg | **-99%** |

### Relação com o Databricks Genie

Este notebook usa a Gemini API como proxy para demonstrar os mesmos conceitos que o **Databricks Genie** aplica nativamente:
- NL → SQL com contexto do Unity Catalog
- Documentação automática de notebooks
- Análise de qualidade via linguagem natural

### Próximos Passos
- [ ] Integração nativa com Databricks via REST API
- [ ] Fine-tuning com terminologia específica do negócio
- [ ] Pipeline de avaliação automática da qualidade dos outputs

---
*Projeto de portfólio por Isac Oliveira · github.com/isac-oliveira-nascimento*